In [1]:
import numpy as np
import pandas as pd
import torch
import random

from Conv1DAE import Conv1DAE, detect_anomalies_conv1dae, conv1dae_train
from optimize_models import latency_to_detection, get_basic_metrics, train_test_split_anomaly_sequence, optimize_conv1dae
from prepare_data import load_or_cache_twitter, load_or_cache_mit_bih, load_or_cache_bonn, \
    get_mit_bih_segments, get_twitter_segments, get_bonn_segments
from visualizations import heatmaps, segments_reconstruction

torch.manual_seed(0)
random.seed(0)

#### <center>Zbiór EEG Bonn</center>

Uznajemy, że zbiór E to outliery, a reszta:
- zdrowi oczy otwarte -> A,
- zdrowi oczy zamknięte -> B,
- pacjenci między napadami (zdrowa półkula) -> C,
- pacjenci między napadami (strefa padaczkowa) -> D,

są zdrowi.

In [2]:
eeg_bonn_dataset = load_or_cache_bonn()

Przygotowany zbiór EEG-Bonn

Każda sekwencja ma przypisaną etykietę na podstawie przynależności do zbioru. Etykieta segmentu jest przypisana na podstawie stosunku liczby anomalii do normalnych próbek na poziomie sekwencji.

In [3]:
bonn_overlaps = [0.25, 0.5, 0.75]
bonn_window_sizes = [2, 3, 5]

<center>Eksperymenty dla 1D Conv-AE</center>

In [4]:
conv1dae_bonn_experiments, conv1dae_bonn_heatmap, conv1dae_bonn_training_params = [
    {
        latent: {
            s: {
                o: None for o in bonn_overlaps
            } for s in bonn_window_sizes
        } for latent in [8, 16]
    }  for _ in range(3)
]

for latent, seconds in conv1dae_bonn_experiments.items():
    for second, overlaps in seconds.items():
        for overlap in overlaps.keys():
            # Przygotowanie danych
            X_bonn, y_bonn, _ = get_bonn_segments(eeg_bonn_dataset, second, overlap)

            # Podział na train/test
            X_train_bonn, X_test_bonn, y_train_bonn, y_test_bonn = train_test_split_anomaly_sequence(X_bonn, y_bonn, random_state=42)

            # Optymalizacja parametrów Conv1DAE
            conv1dae_bonn_study = optimize_conv1dae(
                X=X_train_bonn,
                y=y_train_bonn,
                latent=latent,
                dataset_name="EEG Bonn",
                n_trials=10
            )
            conv1dae_bonn_params = conv1dae_bonn_study.best_params
            conv1dae_bonn_train_params = {k: conv1dae_bonn_params[k] for k in ["lr", "epochs", "alpha", "beta", "gamma"]}
            conv1dae_bonn_loss_params = {k: conv1dae_bonn_params[k] for k in ["alpha", "beta", "gamma"]}

            conv1dae_bonn_training_params[latent][second][overlap] = conv1dae_bonn_params

            # Konwersja z numpy do torch
            X_train_bonn, X_test_bonn = torch.from_numpy(X_train_bonn).to(dtype=torch.float32, device='cuda'), torch.from_numpy(X_test_bonn).to(dtype=torch.float32, device='cuda')

            # Autoenkoder
            conv1dae_bonn = Conv1DAE(input_dim=1, latent_dim=latent)
            X_train_bonn = X_train_bonn[:, :, None, :]
            X_test_bonn = X_test_bonn[:, :, None, :]

            # Trening
            conv1dae_train(
                model=conv1dae_bonn,
                data=X_train_bonn,
                **conv1dae_bonn_train_params
            )

            # Predykcje
            y_pred_bonn, y_scores_bonn, _ , reconstruction_bonn = detect_anomalies_conv1dae(
                model=conv1dae_bonn,
                data=X_test_bonn,
                threshold_percentile=80,
                **conv1dae_bonn_loss_params
            )

            # Zapisanie metryk
            experiment_basic_metrics = get_basic_metrics(
                y_true=y_test_bonn.reshape(-1, 1).squeeze(),
                y_pred=y_pred_bonn.reshape(-1, 1).squeeze(),
                y_scores=y_scores_bonn.reshape(-1, 1).squeeze()
            )
            conv1dae_bonn_test_latencies = latency_to_detection(y_test_bonn, y_pred_bonn)
            conv1dae_bonn_experiments[latent][second][overlap] = {**experiment_basic_metrics, **conv1dae_bonn_test_latencies}
            conv1dae_bonn_heatmap[latent][second][overlap] = y_scores_bonn

            # Rekonstrukcja sygnałów
            segments_reconstruction(
                X_test=X_test_bonn.cpu().numpy().squeeze(2),
                X_pred=reconstruction_bonn,
                y_true=y_test_bonn,
                y_pred=y_pred_bonn.squeeze(),
                title="",
                save_path=f"{latent}_{second}_{int(overlap * 100)}_reconstruction_analysis_bonn.png"
            )

[I 2026-01-18 12:13:03,629] A new study created in memory with name: Optuna for LSTMAE on EEG Bonn dataset


Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3


In [5]:
conv1dae_bonn_experiments

{8: {2: {0.25: {'pr_auc': 0.9793935519798985,
    'f1': 0.6091794158553546,
    'recall': 0.438,
    'detection_rate': 0.65,
    'detected_anomalies': 65,
    'missed_anomalies': 35,
    'total_anomalies': 100,
    'mean_latency': np.float64(1.3384615384615384),
    'median_latency': np.float64(0.0)},
   0.5: {'pr_auc': 0.9788843184158095,
    'f1': 0.6093552465233881,
    'recall': 0.4381818181818182,
    'detection_rate': 0.7,
    'detected_anomalies': 70,
    'missed_anomalies': 30,
    'total_anomalies': 100,
    'mean_latency': np.float64(2.157142857142857),
    'median_latency': np.float64(0.0)},
   0.75: {'pr_auc': 0.9885945213799389,
    'f1': 0.6091354512407144,
    'recall': 0.43795454545454543,
    'detection_rate': 0.66,
    'detected_anomalies': 66,
    'missed_anomalies': 34,
    'total_anomalies': 100,
    'mean_latency': np.float64(2.8484848484848486),
    'median_latency': np.float64(0.0)}},
  3: {0.25: {'pr_auc': 0.9857691420624071,
    'f1': 0.6091794158553546,
    '

In [6]:
conv1dae_bonn_training_params

{8: {2: {0.25: {'alpha': 0.40919616423534183,
    'beta': 0.32613762547568975,
    'gamma': 0.7330880728874676,
    'lr': 0.0015930522616241021,
    'epochs': 43},
   0.5: {'alpha': 0.40919616423534183,
    'beta': 0.32613762547568975,
    'gamma': 0.7330880728874676,
    'lr': 0.0015930522616241021,
    'epochs': 43},
   0.75: {'alpha': 0.38542676439134516,
    'beta': 0.5228296095500715,
    'gamma': 0.3171942605576092,
    'lr': 0.006586289317583112,
    'epochs': 31}},
  3: {0.25: {'alpha': 0.38542676439134516,
    'beta': 0.5228296095500715,
    'gamma': 0.3171942605576092,
    'lr': 0.006586289317583112,
    'epochs': 31},
   0.5: {'alpha': 0.38542676439134516,
    'beta': 0.5228296095500715,
    'gamma': 0.3171942605576092,
    'lr': 0.006586289317583112,
    'epochs': 31},
   0.75: {'alpha': 0.38542676439134516,
    'beta': 0.5228296095500715,
    'gamma': 0.3171942605576092,
    'lr': 0.006586289317583112,
    'epochs': 31}},
  5: {0.25: {'alpha': 0.40919616423534183,
    'bet

In [7]:
for latent in conv1dae_bonn_heatmap.keys():
    heatmaps(conv1dae_bonn_heatmap.get(latent), f"Heatmapy anomalii dla Conv1D-AE z rozmiarem latent = {latent} na zbiorze EEG Bonn", f"{latent}_conv1dae_heatmap_bonn")

#### <center>Zbiór MIT-BIH ECG</center>

Dzięki plikom atr mam dostęp, w którym momencie zostało zarejestrowane uderzenie serca. Dzięki temu mogę każdy z segmentów w sekwencjach oznaczać, ale może być wiele etykiet w segmencie. Aby przypisać czy jest outlierem zliczam wystąpienia "N" i pozostałych i porównuje, czego jest więcej.

In [8]:
mit_bih = load_or_cache_mit_bih()

Przygotowany zbiór MIT BIH

Każdy segment posiada etykietę przypisaną na podstawie stosunku liczby uderzeń normalnych do arytmii.

In [9]:
mit_bih_overlaps = [0.25, 0.5, 0.75]
mit_bih_window_sizes = [7, 10, 15]

<center>Eksperymenty dla 1D Conv-AE</center>

In [10]:
conv1dae_mit_bih_experiments, conv1dae_mit_bih_heatmap, conv1dae_mit_bih_training_params = [
    {
        latent: {
            s: {
                o: None for o in mit_bih_overlaps
            } for s in mit_bih_window_sizes
        } for latent in [8, 16]
    }  for _ in range(3)
]

for latent, seconds in conv1dae_mit_bih_experiments.items():
    for second, overlaps in seconds.items():
        for overlap in overlaps.keys():
            torch.cuda.empty_cache()
            # Przygotowanie danych
            X_mit_bih, y_mit_bih, _ = get_mit_bih_segments(mit_bih, second, overlap)

            # Wybieramy te anomalie, które mają najmniej zanieczyszczonych segmentów
            anomaly_ratios = (y_mit_bih == -1).mean(axis=1)
            pseudo_y_mit_bih = np.where(anomaly_ratios < 0.2, 1, -1)

            # Podział na train/test
            X_train_mit_bih, X_test_mit_bih, y_train_mit_bih, y_test_mit_bih = train_test_split_anomaly_sequence(X_mit_bih, y_mit_bih, pseudo_label=pseudo_y_mit_bih)

            # Optymalizacja parametrów Conv1DAE
            conv1dae_mit_bih_study = optimize_conv1dae(
                X=X_train_mit_bih,
                y=y_train_mit_bih,
                latent=latent,
                dataset_name="MIT BIH",
                n_trials=20,
                percentile=99
            )

            conv1dae_mit_bih_params = conv1dae_mit_bih_study.best_params
            conv1dae_mit_bih_train_params = {k: conv1dae_mit_bih_params[k] for k in ["lr", "epochs", "alpha", "beta", "gamma"]}
            conv1dae_mit_bih_loss_params = {k: conv1dae_mit_bih_params[k] for k in ["alpha", "beta", "gamma"]}

            conv1dae_mit_bih_training_params[latent][second][overlap] = conv1dae_mit_bih_params

            # Konwersja z numpy do torch
            X_train_mit_bih, X_test_mit_bih = torch.from_numpy(X_train_mit_bih).to(dtype=torch.float32, device="cuda"), torch.from_numpy(X_test_mit_bih).to(dtype=torch.float32, device="cuda")

            # Autoenkoder
            conv1dae_mit_bih = Conv1DAE(input_dim=1, latent_dim=latent)
            X_train_mit_bih = X_train_mit_bih[:, :, None, :]
            X_test_mit_bih = X_test_mit_bih[:, :, None, :]

            # Trening
            conv1dae_train(
                model=conv1dae_mit_bih,
                data=X_train_mit_bih,
                **conv1dae_mit_bih_train_params
            )

            # Predykcje
            y_pred_mit_bih, y_scores_mit_bih, _, reconstruction_mit_bih = detect_anomalies_conv1dae(
                model=conv1dae_mit_bih,
                data=X_test_mit_bih,
                threshold_percentile=95,
                **conv1dae_mit_bih_loss_params
            )

            # Zapisanie metryk
            experiment_basic_metrics = get_basic_metrics(
                y_test_mit_bih.reshape(-1, 1).squeeze(),
                y_pred_mit_bih.reshape(-1, 1).squeeze(),
                y_scores_mit_bih.reshape(-1, 1).squeeze()
            )
            conv1dae_mit_bih_test_latencies = latency_to_detection(y_test_mit_bih, y_pred_mit_bih)
            conv1dae_mit_bih_experiments[latent][second][overlap] = {**experiment_basic_metrics, **conv1dae_mit_bih_test_latencies}
            conv1dae_mit_bih_heatmap[latent][second][overlap] = y_scores_mit_bih

            # Rekonstrukcja sygnałów
            segments_reconstruction(
                X_test=X_test_mit_bih.cpu().numpy().squeeze(2),
                X_pred=reconstruction_mit_bih,
                y_true=y_test_mit_bih,
                y_pred=y_pred_mit_bih.squeeze(),
                n=2,
                title="",
                save_path=f"{latent}_{second}_{int(overlap * 100)}_reconstruction_analysis_mit_bih.png"
            )

Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3
Zbyt mały test_size. Nowa wartość = 0.3


In [11]:
conv1dae_mit_bih_experiments

{8: {7: {0.25: {'pr_auc': 0.3524248882698296,
    'f1': 0.19699595609474294,
    'recall': 0.11918909472212513,
    'detection_rate': 0.10028653295128939,
    'detected_anomalies': 35,
    'missed_anomalies': 314,
    'total_anomalies': 349,
    'mean_latency': np.float64(4.314285714285714),
    'median_latency': np.float64(1.0)},
   0.5: {'pr_auc': 0.3511499333728034,
    'f1': 0.22777454968041835,
    'recall': 0.13709489391466542,
    'detection_rate': 0.1024390243902439,
    'detected_anomalies': 42,
    'missed_anomalies': 368,
    'total_anomalies': 410,
    'mean_latency': np.float64(4.5476190476190474),
    'median_latency': np.float64(1.0)},
   0.75: {'pr_auc': 0.37506384952499805,
    'f1': 0.23350644354683592,
    'recall': 0.14117920688452146,
    'detection_rate': 0.13446969696969696,
    'detected_anomalies': 71,
    'missed_anomalies': 457,
    'total_anomalies': 528,
    'mean_latency': np.float64(5.098591549295775),
    'median_latency': np.float64(0.0)}},
  10: {0.25:

In [12]:
conv1dae_mit_bih_training_params

{8: {7: {0.25: {'alpha': 0.4574882495833371,
    'beta': 0.30388213415932674,
    'gamma': 0.645406737731335,
    'lr': 0.0038426938214315607,
    'epochs': 45},
   0.5: {'alpha': 0.38542676439134516,
    'beta': 0.5228296095500715,
    'gamma': 0.3171942605576092,
    'lr': 0.006586289317583112,
    'epochs': 31},
   0.75: {'alpha': 0.38542676439134516,
    'beta': 0.5228296095500715,
    'gamma': 0.3171942605576092,
    'lr': 0.006586289317583112,
    'epochs': 31}},
  10: {0.25: {'alpha': 0.4574882495833371,
    'beta': 0.30388213415932674,
    'gamma': 0.645406737731335,
    'lr': 0.0038426938214315607,
    'epochs': 45},
   0.5: {'alpha': 0.48705288056258855,
    'beta': 0.30920676736434793,
    'gamma': 0.637068317340386,
    'lr': 0.002389525370602796,
    'epochs': 46},
   0.75: {'alpha': 0.7252813963310067,
    'beta': 0.3767358556592812,
    'gamma': 0.33252579649263975,
    'lr': 0.007902619549708232,
    'epochs': 50}},
  15: {0.25: {'alpha': 0.606656649223281,
    'beta': 

In [13]:
for latent in conv1dae_mit_bih_heatmap.keys():
    heatmaps(conv1dae_mit_bih_heatmap.get(latent), f"Heatmapy anomalii dla Conv1D-AE z rozmiarem latent = {latent} na zbiorze MIT BIH", f"{latent}_conv1dae_heatmap_mit_bih")

#### <center>Zbiór Numenta Anomaly Benchmark (Twitter)</center>

In [14]:
twitter = load_or_cache_twitter()

Przygotowany zbiór Twitter

Każdy segment posiada etykietę przypisaną na podstawie stosunku liczby anomalii do liczby prawidłowych próbek.

In [15]:
twitter_window_sizes = [60 * 1, 60 * 2, 60 * 3]
twitter_overlaps = [0.25, 0.5, 0.75]

<center>Eksperymenty dla 1D Conv-AE</center>

In [16]:
conv1dae_twitter_experiments, conv1dae_twitter_heatmap, conv1dae_twitter_training_params = [
    {
        latent: {
            s: {
                o: None for o in twitter_overlaps
            } for s in twitter_window_sizes
        } for latent in [8, 16]
    }  for _ in range(3)
]

for latent, seconds in conv1dae_twitter_experiments.items():
    for second, overlaps in seconds.items():
        for overlap in overlaps.keys():
            # Przygotowanie danych
            X_twitter, y_twitter, _ = get_twitter_segments(twitter, second, overlap)

            # Podział na train/test
            X_train_twitter, X_test_twitter, y_train_twitter, y_test_twitter = train_test_split_anomaly_sequence(X_twitter, y_twitter, random_state=42)

            # Optymalizacja parametrów Conv1DAE
            conv1dae_twitter_study = optimize_conv1dae(
                X=X_train_twitter,
                y=y_train_twitter,
                latent=latent,
                dataset_name="Twitter",
                n_trials=100,
                percentile=99
            )
            conv1dae_twitter_params = conv1dae_twitter_study.best_params
            conv1dae_twitter_train_params = {k: conv1dae_twitter_params[k] for k in ["lr", "epochs", "alpha", "beta", "gamma"]}
            conv1dae_twitter_loss_params = {k: conv1dae_twitter_params[k] for k in ["alpha", "beta", "gamma"]}
            conv1dae_twitter_training_params[latent][second][overlap] = conv1dae_twitter_params

            # Konwersja z numpy do torch
            X_train_twitter, X_test_twitter = torch.from_numpy(X_train_twitter).to(dtype=torch.float32, device="cuda"), torch.from_numpy(X_test_twitter).to(dtype=torch.float32, device="cuda")

            # Autoenkoder
            conv1dae_twitter = Conv1DAE(input_dim=1, latent_dim=latent)
            X_train_twitter = X_train_twitter[:, :, None, :]
            X_test_twitter = X_test_twitter[:, :, None, :]

            # Trening
            conv1dae_train(
                model=conv1dae_twitter,
                data=X_train_twitter,
                **conv1dae_twitter_train_params
            )

            # Predykcje
            y_pred_twitter, y_scores_twitter, _ ,reconstruction_twitter = detect_anomalies_conv1dae(
                model=conv1dae_twitter,
                data=X_test_twitter,
                **conv1dae_twitter_loss_params
            )

            # Zapisanie metryk
            experiment_basic_metrics = get_basic_metrics(
                y_test_twitter.reshape(-1, 1).squeeze(),
                y_pred_twitter.reshape(-1, 1).squeeze(),
                y_scores_twitter.reshape(-1, 1).squeeze()
            )
            conv1dae_twitter_test_latencies = latency_to_detection(y_test_twitter, y_pred_twitter)
            conv1dae_twitter_experiments[latent][second][overlap] = {**experiment_basic_metrics, **conv1dae_twitter_test_latencies}
            conv1dae_twitter_heatmap[latent][second][overlap] = y_scores_twitter

            # Rekonstrukcja sygnałów
            segments_reconstruction(
                X_test=X_test_twitter.cpu().numpy().squeeze(2),
                X_pred=reconstruction_twitter,
                y_true=y_test_twitter,
                y_pred=y_pred_twitter.squeeze(),
                n=2,
                title="",
                save_path=f"{latent}_{second}_{int(overlap * 100)}_reconstruction_analysis_twitter.png"
            )

Zbyt mały test_size. Nowa wartość = 0.6454545454545454
Zbyt mały test_size. Nowa wartość = 0.6454545454545454
Zbyt mały test_size. Nowa wartość = 0.6454545454545454
Zbyt mały test_size. Nowa wartość = 0.6454545454545454
Zbyt mały test_size. Nowa wartość = 0.6454545454545454
Zbyt mały test_size. Nowa wartość = 0.6454545454545454
Zbyt mały test_size. Nowa wartość = 0.6454545454545454
Zbyt mały test_size. Nowa wartość = 0.6454545454545454
Zbyt mały test_size. Nowa wartość = 0.6454545454545454
Zbyt mały test_size. Nowa wartość = 0.6454545454545454
Zbyt mały test_size. Nowa wartość = 0.6454545454545454
Zbyt mały test_size. Nowa wartość = 0.6454545454545454
Zbyt mały test_size. Nowa wartość = 0.6454545454545454
Zbyt mały test_size. Nowa wartość = 0.6454545454545454
Zbyt mały test_size. Nowa wartość = 0.6454545454545454
Zbyt mały test_size. Nowa wartość = 0.6454545454545454
Zbyt mały test_size. Nowa wartość = 0.6454545454545454
Zbyt mały test_size. Nowa wartość = 0.6454545454545454


In [17]:
conv1dae_twitter_experiments

{8: {60: {0.25: {'pr_auc': 0.06632628396030846,
    'f1': 0.02569593147751606,
    'recall': 0.022641509433962263,
    'detection_rate': 0.3333333333333333,
    'detected_anomalies': 2,
    'missed_anomalies': 4,
    'total_anomalies': 6,
    'mean_latency': np.float64(11.0),
    'median_latency': np.float64(11.0)},
   0.5: {'pr_auc': 0.06584918598450401,
    'f1': 0.03125,
    'recall': 0.02736318407960199,
    'detection_rate': 0.3333333333333333,
    'detected_anomalies': 2,
    'missed_anomalies': 4,
    'total_anomalies': 6,
    'mean_latency': np.float64(16.5),
    'median_latency': np.float64(16.5)},
   0.75: {'pr_auc': 0.07368524147161416,
    'f1': 0.03409090909090909,
    'recall': 0.029850746268656716,
    'detection_rate': 0.6666666666666666,
    'detected_anomalies': 4,
    'missed_anomalies': 2,
    'total_anomalies': 6,
    'mean_latency': np.float64(5.0),
    'median_latency': np.float64(5.0)}},
  120: {0.25: {'pr_auc': 0.11173775927935259,
    'f1': 0.08583690987124463

In [18]:
conv1dae_twitter_training_params

{8: {60: {0.25: {'alpha': 0.32900254056032374,
    'beta': 0.36679138114509496,
    'gamma': 0.3186112045481629,
    'lr': 0.008349622718389453,
    'epochs': 49},
   0.5: {'alpha': 0.41854784791336364,
    'beta': 0.5356817513864743,
    'gamma': 0.3458163137357676,
    'lr': 0.006574235265536276,
    'epochs': 48},
   0.75: {'alpha': 0.33343564988596575,
    'beta': 0.526140219302886,
    'gamma': 0.3713149826486129,
    'lr': 0.00775369463661958,
    'epochs': 46}},
  120: {0.25: {'alpha': 0.31693796392764473,
    'beta': 0.5403913276660265,
    'gamma': 0.35071607441188457,
    'lr': 0.008522169197479671,
    'epochs': 33},
   0.5: {'alpha': 0.3245166255404477,
    'beta': 0.3900210555915658,
    'gamma': 0.36796611957719727,
    'lr': 0.005526333639589194,
    'epochs': 49},
   0.75: {'alpha': 0.32515665081988543,
    'beta': 0.4328769284411865,
    'gamma': 0.36557730572875047,
    'lr': 0.008674208596137156,
    'epochs': 40}},
  180: {0.25: {'alpha': 0.31630441462869685,
    'b

Heatmapy

In [19]:
for latent in conv1dae_twitter_heatmap.keys():
    heatmaps(conv1dae_twitter_heatmap.get(latent), f"Heatmapy anomalii dla Conv1D-AE z rozmiarem latent = {latent} na zbiorze Twitter", f"{latent}_conv1dae_heatmap_twitter")